# Solving a 2D Maze with an Energy-Based Model (THRML)

Every maze cell is a 3-state categorical variable — **Path**, **NotPath**, or **Wall** —
and the maze is encoded purely in the energy function:

$$E(s) = -\beta \Big( \sum_i b_i[s_i] \; + \sum_{(u,v)} W[s_u, s_v] \Big)$$

- **Biases** $b_i$: wall cells prefer Wall, start/end prefer Path, corridor cells are split between Path and NotPath. Off-grid neighbors of border cells count as Walls (padding), folded into the border biases.
- **Edge function** $W$ (3x3): couples each pair of 4-adjacent cells, applied once per edge in row-major scan order.
- **Sampling**: block Gibbs over the two bipartite (checkerboard) blocks of the grid graph, using THRML's `CategoricalEBMFactor` + `CategoricalGibbsConditional` + `FactorSamplingProgram`.

Low-energy states are configurations where the Path cells trace a route from start to end.
All logic lives in [`utils/`](utils): `config.py` (every tunable), `wrapper_maze.py` (mazelib → THRML graph), `model.py` (factors, sampling, diagnostics), `visualize.py` (plots).

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import jax
import matplotlib.pyplot as plt
import numpy as np
from thrml import SamplingSchedule

from utils import config, model, visualize
from utils.wrapper_maze import generate_maze, maze_to_graph

## 1. Configuration

Everything tunable is in `utils/config.py`. The canonical state order is
**(Path, NotPath, Wall) = (0, 1, 2)**, matching the row/col order of the edge matrix.
Positive entries are *favorable* (the energy is minus the bias/weight).

In [ ]:
print('State order :', config.STATE_NAMES)
print('Bias wall   :', config.BIAS_WALL)
print('Bias goal   :', config.BIAS_GOAL)
print('Bias other  :', config.BIAS_OTHER)
print('Edge weights W (rows/cols = Path, NotPath, Wall):')
print(config.EDGE_WEIGHTS)
print('beta        :', config.BETA)
print(f'Maze        : {config.MAZE_WIDTH}x{config.MAZE_HEIGHT} cells, seed {config.SEED}')
print(f'Sampling    : warmup={config.N_WARMUP}, samples={config.N_SAMPLES}, '
      f'steps/sample={config.STEPS_PER_SAMPLE}, chains={config.N_CHAINS}')

## 2. Generate a maze with mazelib

`mazelib` (Prim's algorithm) produces a `(2H+1, 2W+1)` 0/1 grid — walls are cells too,
including the outer ring — plus start/end openings on the border.

In [ ]:
grid, start, end = generate_maze()
print('grid shape:', grid.shape, '| start:', start, '| end:', end)
visualize.plot_maze(grid, start, end);

## 3. Map the maze to our THRML format

`maze_to_graph` builds the static THRML structure: one `CategoricalNode` per cell,
the two bipartite blocks (checkerboard — no two neighbors update simultaneously),
the scan-order edge list, and the bias array (cell type + wall padding on the border).

In [ ]:
mg = maze_to_graph(grid, start, end)
print('nodes:', len(mg.nodes), '| edges:', len(mg.edges))
print('bipartite block sizes:', [len(b.nodes) for b in mg.blocks])

fig, axes = plt.subplots(1, config.N_STATES, figsize=(15, 4.6))
vmin, vmax = mg.biases.min(), mg.biases.max()
for k, ax in enumerate(axes):
    im = ax.imshow(mg.biases[:, k].reshape(grid.shape), cmap='magma', vmin=vmin, vmax=vmax)
    ax.set_title(f'bias toward {config.STATE_NAMES[k]}')
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, fraction=0.02)
fig.suptitle('Per-cell bias b_i, one panel per state '
             '(goals in the Path panel, walls in the Wall panel, border padding visible)');

## 4. Build the sampling program

Two `CategoricalEBMFactor`s realize the energy — a bias factor with weights `(n_nodes, 3)`
and a pairwise factor with weights `(n_edges, 3, 3)` — assembled into a
`FactorSamplingProgram` with one `CategoricalGibbsConditional(3)` per block.

In [ ]:
program = model.build_program(mg)
print('Free blocks         :', [len(b.nodes) for b in mg.blocks])
print('Bias factor weights :', (len(mg.nodes), config.N_STATES))
print('Edge factor weights :', (len(mg.edges), config.N_STATES, config.N_STATES))

## 5. Correctness check: sampler vs exact Boltzmann distribution

Sampling code fails quietly, so before trusting the maze results we validate on a tiny
1x6 open corridor (6 nodes, $3^6 = 729$ states): enumerate every state, compute its exact
Boltzmann probability $p \propto e^{-E}$ from the *same* energy code, and compare with the
empirical histogram of a long Gibbs run. A wrong sign, edge orientation, or padding bug
would show up as a systematic mismatch here.

In [ ]:
corridor = np.zeros((1, 6), dtype=int)
mg_tiny = maze_to_graph(corridor, (0, 0), (0, 5))
n_tiny = len(mg_tiny.nodes)

all_states = np.array(list(itertools.product(range(config.N_STATES), repeat=n_tiny)))
E_exact = model.energy_of_states(mg_tiny, all_states)
p_exact = np.exp(-(E_exact - E_exact.min()))
p_exact /= p_exact.sum()

prog_tiny = model.build_program(mg_tiny)
sched_tiny = SamplingSchedule(n_warmup=1000, n_samples=2000, steps_per_sample=5)
samples_tiny = model.run_sampling(jax.random.key(1), mg_tiny, prog_tiny, sched_tiny, n_chains=64)

powers = config.N_STATES ** np.arange(n_tiny)
codes = (samples_tiny.reshape(-1, n_tiny) * powers).sum(-1)
exact_codes = (all_states * powers).sum(-1)
p_emp = np.bincount(codes, minlength=config.N_STATES ** n_tiny)[exact_codes] / len(codes)

tvd = 0.5 * np.abs(p_emp - p_exact).sum()
print(f'total variation distance (exact vs empirical): {tvd:.4f}')
assert tvd < 0.05, 'sampler disagrees with the exact Boltzmann distribution'

top = np.argsort(p_exact)[::-1][:30]
plt.figure(figsize=(9, 3.5))
plt.bar(np.arange(len(top)) - 0.2, p_exact[top], width=0.4, label='exact')
plt.bar(np.arange(len(top)) + 0.2, p_emp[top], width=0.4, label='empirical')
plt.xlabel('state (top 30 by exact probability)')
plt.ylabel('probability')
plt.title('Exact Boltzmann vs Gibbs sampler, 1x6 corridor')
plt.legend();

## 6. Sample the maze EBM

Run independent parallel chains (`jax.vmap`) from uniform-random initial states.
Each recorded sample is one full sweep over both blocks (A then B).

In [ ]:
schedule = SamplingSchedule(n_warmup=config.N_WARMUP, n_samples=config.N_SAMPLES,
                            steps_per_sample=config.STEPS_PER_SAMPLE)
states = model.run_sampling(jax.random.key(config.SEED), mg, program, schedule)
print('sampled states:', states.shape, '(chains, samples, nodes)')

### Evolution of one chain

Watch the wall cells (dark): when the wall bias is weaker than the edge energy a wall can
gain by flipping (up to ~8, see section 7), walls *erode* over the sweeps — with the
original spec (wall bias 1) they dissolve entirely within a few sweeps.

In [ ]:
visualize.plot_sampling_evolution(mg, states[0],
                                  [0, 1, 2, 5, 10, 50, config.N_SAMPLES - 1]);

### Energy convergence

In [ ]:
energies = model.energy_of_states(mg, states)
visualize.plot_energy_trace(energies);

### Path marginals and the lowest-energy state

Left: empirical $P(s_i = \text{Path})$ per cell over the second half of every chain.
Right: the single lowest-energy state found across all chains and samples.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
second_half = states[:, states.shape[1] // 2:]
visualize.plot_path_marginal(mg, second_half, ax=axes[0])
best_chain, best_t = np.unravel_index(np.argmin(energies), energies.shape)
visualize.plot_state(mg, states[best_chain, best_t], ax=axes[1],
                     title=f'Lowest-energy state (chain {best_chain}, sample {best_t}, '
                           f'E={energies[best_chain, best_t]:.1f})');

## 7. How many steps does it take?

A chain counts as **solved** when its Path-labeled cells 4-connect start to end
(`model.is_solved`). We track the fraction of solved chains per sample step and the
first step at which each chain solves.

In [ ]:
solved = model.solved_matrix(mg, states)
visualize.plot_solved_fraction(solved)

ever = solved.any(axis=1)
first = np.where(ever, solved.argmax(axis=1), -1)
print(f'chains solved at least once : {ever.sum()}/{len(ever)}')
if ever.any():
    print(f'first-solved sample step    : median {np.median(first[ever]):.0f}, '
          f'min {first[ever].min()}, max {first[ever].max()}')
print(f'solved in final sample      : {solved[:, -1].mean():.0%} of chains')

**Why walls erode when their bias is weak.** A wall cell in state Wall earns only its bias
and contributes nothing through its edges (the Wall row of $W$ is zero), while its mere
presence *penalizes* each Path/NotPath neighbor by $-1$ (the Wall column). Flipping that same
cell to Path instead earns up to $+1$ per edge from $W[P,P]$ — up to $\approx 8$ energy
across four edges. Whenever the wall bias in `config.py` is smaller than that, the global
minimum is *everything Path*: the maze is "solved" by flooding, walls included. The sampler
is provably faithful to this energy (section 5) — the energy itself simply prefers no walls.
The next section uses the modular config to make walls hold firmly.

## 8. Playground: tweak the configuration

All parameters are keyword overrides — no need to edit `config.py`. To make walls hold, the
wall bias must beat the $\approx 8$ energy a wall can gain by melting into Path, so we raise
it to 10. We also cool the distribution ($\beta = 4$) so corridors freeze into their all-Path
minimum, strengthen the goal pull, and give the chain more sweeps between recorded samples.

In [ ]:
custom_bias_wall = np.array([0.0, 0.0, 10.0])  # walls must out-bid ~8 of edge energy
custom_bias_goal = np.array([3.0, 0.0, 0.0])   # stronger Path pull at start/end
custom_beta = 4.0                               # cold: corridors freeze toward Path
custom_schedule = SamplingSchedule(n_warmup=0, n_samples=config.N_SAMPLES,
                                   steps_per_sample=5)

mg_c = maze_to_graph(grid, start, end, bias_wall=custom_bias_wall,
                     bias_goal=custom_bias_goal)
prog_c = model.build_program(mg_c, beta=custom_beta)
states_c = model.run_sampling(jax.random.key(0), mg_c, prog_c, custom_schedule)
energies_c = model.energy_of_states(mg_c, states_c, beta=custom_beta)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
visualize.plot_path_marginal(mg_c, states_c[:, states_c.shape[1] // 2:], ax=axes[0],
                             title=f'P(Path), beta={custom_beta}, wall bias 10')
best = np.unravel_index(np.argmin(energies_c), energies_c.shape)
visualize.plot_state(mg_c, states_c[best], ax=axes[1],
                     title=f'Lowest-energy state (E={energies_c[best]:.1f})')

wall_cells = (grid == 1).reshape(-1)
print('wall cells still Wall in final samples:',
      f'{(states_c[:, -1][:, wall_cells] == config.WALL).mean():.0%}')

solved_c = model.solved_matrix(mg_c, states_c)
ever_c = solved_c.any(axis=1)
first_c = np.where(ever_c, solved_c.argmax(axis=1), -1)
print(f'chains solved at least once: {ever_c.sum()}/{len(ever_c)} '
      f'(median first-solved step {np.median(first_c[ever_c]):.0f})')
print(f'solved in final sample: {solved_c[:, -1].mean():.0%} '
      '(connectivity flickers at finite temperature: one corridor cell flipping '
      'anywhere along the route breaks it)')

## Takeaways

- The maze lives entirely in the energy function: biases pin walls and goals, the 3x3 edge
  function shapes how Path/NotPath domains interact, and block Gibbs sampling relaxes the
  grid into low-energy states.
- The bipartite 2-coloring of the grid is the minimal valid block structure: each sweep
  updates half the cells in parallel, which is what makes this map onto probabilistic hardware.
- The enumeration check in section 5 is the load-bearing test — it pins the sampler to the
  exact Boltzmann distribution of this energy, signs and orientations included.
- Energy design matters as much as sampling: a wall bias below the ~8 of edge energy a wall
  gains by melting into Path lets walls erode; raising it to 10 and cooling to $\beta=4$
  restores the maze, and then every chain finds a start-to-end route.
- Every knob (biases, edge matrix, beta, schedule) is a `config.py` default with a keyword
  override, so experimenting with different energy designs is a one-cell change.